In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

df = pd.read_csv('../data/processed/spain_top50_clean.csv')
df['date'] = pd.to_datetime(df['date'])

print(f"Rows: {len(df)}")
print(f"Unique dates: {df['date'].nunique()}")
print(f"Unique songs (song_key + artist_key): {df.groupby(['song_key','artist_key']).ngroups}")
df.head()

Rows: 27750
Unique dates: 555
Unique songs (song_key + artist_key): 575


,date,position,song,artist,popularity,duration_ms,album_type,total_tracks,is_explicit,album_cover_url,song_key,artist_key
0,2024-05-18,1,Supernova,Saiko,84,190909,album,16,False,https://i.scdn.co/image/ab67616d0000b273e346fc...,supernova,saiko
1,2024-05-18,2,EL TONTO,Lola Indigo,83,187986,album,11,False,https://i.scdn.co/image/ab67616d0000b273daab36...,el tonto,lola indigo
2,2024-05-18,3,Acróstico,Shakira,81,170785,single,1,False,https://i.scdn.co/image/ab67616d0000b273bd6bbd...,acróstico,shakira
3,2024-05-18,4,BESO,ROSALÍA & Rauw Alejandro,96,194543,single,3,False,https://i.scdn.co/image/ab67616d0000b2734d6cf0...,beso,rosalía & rauw alejandro
4,2024-05-18,5,El Merengue,Marshmello & Manuel Turizo,91,189668,single,1,False,https://i.scdn.co/image/ab67616d0000b273f40467...,el merengue,marshmello & manuel turizo


In [3]:
lifecycle = (
    df.groupby(['song_key', 'artist_key'])
    .agg(
        song=('song', 'first'),           # keep one display name (first alphabetically by date order)
        artist=('artist', 'first'),
        entry_date=('date', 'min'),
        exit_date=('date', 'max'),
        peak_position=('position', 'min'),  # lower number = higher chart position
        total_appearances=('date', 'count'),  # raw row count, for comparison
        album_type=('album_type', 'first'),
        is_explicit=('is_explicit', 'first'),
        total_tracks=('total_tracks', 'first'),
    )
    .reset_index()
)

lifecycle['days_on_chart'] = (lifecycle['exit_date'] - lifecycle['entry_date']).dt.days + 1

print(f"Total songs in lifecycle table: {len(lifecycle)}")
lifecycle.sort_values('days_on_chart', ascending=False).head(10)

Total songs in lifecycle table: 575


,song_key,artist_key,song,artist,entry_date,exit_date,peak_position,total_appearances,album_type,is_explicit,total_tracks,days_on_chart
265,la bachata,manuel turizo,La Bachata,Manuel Turizo,2024-05-18,2025-11-19,11,538,album,False,15,551
29,amargura,karol g,AMARGURA,KAROL G,2024-05-26,2025-10-31,16,153,album,True,17,524
546,vagabundo,sebastian yatra & manuel turizo & beéle,VAGABUNDO,Sebastian Yatra & Manuel Turizo & Beéle,2024-05-18,2025-10-16,3,385,single,False,1,517
97,columbia,quevedo,Columbia,Quevedo,2024-07-08,2025-11-02,1,463,single,False,1,483
293,lala,myke towers,LALA,Myke Towers,2024-06-29,2025-09-24,1,310,album,True,23,453
524,tqg,karol g,TQG,KAROL G,2024-05-18,2025-08-08,9,120,album,True,17,448
429,qlona,karol g,QLONA,KAROL G,2024-08-19,2025-10-19,8,342,album,True,9,427
26,amanece,anuel aa & haze,Amanece,Anuel AA & Haze,2024-05-25,2025-07-06,47,15,single,False,1,408
167,el merengue,marshmello,El Merengue,Marshmello,2024-11-04,2025-11-18,19,317,album,False,11,380
325,manos rotas,dellafuente & morad,Manos Rotas,DELLAFUENTE & Morad,2024-11-18,2025-11-23,1,329,single,False,1,371


In [4]:
lifecycle = lifecycle.rename(columns={
    'days_on_chart': 'chart_span_days',      # entry to exit, calendar span
    'total_appearances': 'days_on_chart'     # days it actually appeared in the Top 50
})
lifecycle['days_off_chart'] = lifecycle['chart_span_days'] - lifecycle['days_on_chart']

print(f"Songs with at least one day off-chart inside their span: {(lifecycle['days_off_chart'] > 0).sum()} of {len(lifecycle)}")
lifecycle.sort_values('days_on_chart', ascending=False).head(10)[
    ['song', 'artist', 'entry_date', 'exit_date', 'chart_span_days', 'days_on_chart', 'days_off_chart']
]

Songs with at least one day off-chart inside their span: 240 of 575


,song,artist,entry_date,exit_date,chart_span_days,days_on_chart,days_off_chart
265,La Bachata,Manuel Turizo,2024-05-18,2025-11-19,551,538,13
97,Columbia,Quevedo,2024-07-08,2025-11-02,483,463,20
546,VAGABUNDO,Sebastian Yatra & Manuel Turizo & Beéle,2024-05-18,2025-10-16,517,385,132
269,LA FALDA,Myke Towers,2024-11-25,2025-11-23,364,359,5
311,LUNA,Feid,2024-12-02,2025-11-27,361,357,4
429,QLONA,KAROL G,2024-08-19,2025-10-19,427,342,85
325,Manos Rotas,DELLAFUENTE & Morad,2024-11-18,2025-11-23,371,329,42
167,El Merengue,Marshmello,2024-11-04,2025-11-18,380,317,63
417,POLARIS - Remix,Saiko & Feid & Quevedo,2024-06-11,2025-04-20,314,312,2
293,LALA,Myke Towers,2024-06-29,2025-09-24,453,310,143


In [5]:
# Number each date by its position among the dates we actually have
date_index = {d: i for i, d in enumerate(sorted(df['date'].unique()))}
df['date_idx'] = df['date'].map(date_index)

# Sort so each song's appearances are in order
df = df.sort_values(['song_key', 'artist_key', 'date_idx'])

# Gap between a song's consecutive appearances (NaN for its first appearance)
gap = df.groupby(['song_key', 'artist_key'])['date_idx'].diff()

# A new run starts at the first appearance or whenever the gap is not exactly 1
df['new_run'] = gap != 1
df['run_id'] = df.groupby(['song_key', 'artist_key'])['new_run'].cumsum()

# Count runs per song and attach to the lifecycle table
runs_per_song = (
    df.groupby(['song_key', 'artist_key'])['run_id'].max()
    .rename('num_runs')
    .reset_index()
)
lifecycle = lifecycle.merge(runs_per_song, on=['song_key', 'artist_key'], how='left')

print(lifecycle['num_runs'].value_counts().sort_index())

num_runs
1     382
2     107
3      45
4      14
5       6
6       8
7       2
8       5
9       1
11      1
13      1
14      1
15      1
22      1
Name: count, dtype: int64


In [6]:
group_cols = ['song_key', 'artist_key']

# First date each song reached its best (lowest-number) position
is_peak_row = df['position'] == df.groupby(group_cols)['position'].transform('min')
peak_dates = (
    df[is_peak_row]
    .groupby(group_cols)['date'].min()
    .rename('peak_date')
    .reset_index()
)
lifecycle = lifecycle.merge(peak_dates, on=group_cols, how='left')

lifecycle['time_to_peak_days'] = (lifecycle['peak_date'] - lifecycle['entry_date']).dt.days

# Flags for songs whose true entry/exit falls outside our data window
lifecycle['entered_before_data'] = lifecycle['entry_date'] == df['date'].min()
lifecycle['still_on_chart'] = lifecycle['exit_date'] == df['date'].max()

print(f"Songs already charting on day one of the data: {lifecycle['entered_before_data'].sum()}")
print(f"Songs still charting on the last day: {lifecycle['still_on_chart'].sum()}")
print()
print(lifecycle['time_to_peak_days'].describe())

Songs already charting on day one of the data: 50
Songs still charting on the last day: 50

count    575.000000
mean       9.483478
std       30.019436
min        0.000000
25%        0.000000
50%        0.000000
75%        7.500000
max      426.000000
Name: time_to_peak_days, dtype: float64


In [7]:
real_debuts = lifecycle[~lifecycle['entered_before_data']]

print(f"Songs with an observed entry: {len(real_debuts)}")
print(f"Share peaking on entry day: {(real_debuts['time_to_peak_days'] == 0).mean():.1%}")
print()
print(real_debuts['time_to_peak_days'].describe())
print()
print("Slowest risers:")
real_debuts.sort_values('time_to_peak_days', ascending=False).head(8)[
    ['song', 'artist', 'entry_date', 'peak_date', 'peak_position', 'time_to_peak_days', 'num_runs']
]

Songs with an observed entry: 525
Share peaking on entry day: 52.0%

count    525.000000
mean       9.598095
std       29.829803
min        0.000000
25%        0.000000
50%        0.000000
75%        8.000000
max      426.000000
Name: time_to_peak_days, dtype: float64

Slowest risers:


,song,artist,entry_date,peak_date,peak_position,time_to_peak_days,num_runs
29,AMARGURA,KAROL G,2024-05-26,2025-07-26,16,426,4
342,Mi Luz (ft. Rels B),Rvfv,2024-10-23,2025-10-11,32,353,15
14,A 300,JC Reyes & Pedro Calderon,2024-08-06,2025-01-05,15,152,2
290,La_Original.mp3,Emilia,2024-11-04,2025-03-08,10,124,3
118,COSAS QUE NO TE DIJE,Saiko,2024-08-16,2024-12-12,7,118,1
156,Déjame Entrar,Rauw Alejandro,2025-07-25,2025-11-17,17,115,3
317,Madrid City,Ana Mena,2024-09-30,2025-01-03,13,95,2
269,LA FALDA,Myke Towers,2024-11-25,2025-02-18,1,85,2


In [8]:
# Only each song's first unbroken streak on the chart
first_run = df[df['run_id'] == 1]
best_in_first_run = first_run.groupby(group_cols)['position'].transform('min')

first_peak = (
    first_run[first_run['position'] == best_in_first_run]
    .groupby(group_cols)
    .agg(first_run_peak_position=('position', 'min'),
         first_run_peak_date=('date', 'min'))
    .reset_index()
)
lifecycle = lifecycle.merge(first_peak, on=group_cols, how='left')
lifecycle['time_to_peak_first_run'] = (
    lifecycle['first_run_peak_date'] - lifecycle['entry_date']
).dt.days

real_debuts = lifecycle[~lifecycle['entered_before_data']]
print(f"Share peaking on entry day: {(real_debuts['time_to_peak_first_run'] == 0).mean():.1%}")
print()
print(real_debuts['time_to_peak_first_run'].describe())
print()
print("Slowest risers (first run only):")
real_debuts.sort_values('time_to_peak_first_run', ascending=False).head(8)[
    ['song', 'artist', 'entry_date', 'first_run_peak_date', 'first_run_peak_position', 'time_to_peak_first_run']
]

Share peaking on entry day: 59.0%

count    525.000000
mean       5.274286
std       13.180956
min        0.000000
25%        0.000000
50%        0.000000
75%        4.000000
max      118.000000
Name: time_to_peak_first_run, dtype: float64

Slowest risers (first run only):


,song,artist,entry_date,first_run_peak_date,first_run_peak_position,time_to_peak_first_run
118,COSAS QUE NO TE DIJE,Saiko,2024-08-16,2024-12-12,7,118
361,Narcotics (with Bryant Myers),Dei V & Bryant Myers,2024-10-04,2024-12-28,15,85
269,LA FALDA,Myke Towers,2024-11-25,2025-02-18,1,85
377,No_se_ve.mp3,Emilia & LUDMILLA & ZECCA,2024-06-08,2024-08-25,13,78
37,Arena y Sal,Omar Montes & Saiko & Tunvao,2024-05-21,2024-08-06,3,77
313,Luna llena,Beny Jr,2025-07-31,2025-10-14,4,75
371,NO PIENSO LLAMAR,Soge Culebra,2024-10-14,2024-12-23,7,70
352,MODELITO,Mora,2024-09-09,2024-11-12,4,64


In [9]:
df = df.drop(columns=['new_run'])  # helper column, no longer needed

lifecycle.to_csv('../data/processed/lifecycle_table.csv', index=False)
df.to_csv('../data/processed/spain_top50_with_runs.csv', index=False)

print(f"lifecycle_table.csv: {lifecycle.shape}")
print(f"spain_top50_with_runs.csv: {df.shape}")
print()
print(list(lifecycle.columns))

lifecycle_table.csv: (575, 21)
spain_top50_with_runs.csv: (27750, 14)

['song_key', 'artist_key', 'song', 'artist', 'entry_date', 'exit_date', 'peak_position', 'days_on_chart', 'album_type', 'is_explicit', 'total_tracks', 'chart_span_days', 'days_off_chart', 'num_runs', 'peak_date', 'time_to_peak_days', 'entered_before_data', 'still_on_chart', 'first_run_peak_position', 'first_run_peak_date', 'time_to_peak_first_run']
